In [0]:
import os, subprocess
_nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
_req = f"/Workspace{os.path.dirname(_nb)}/requirements-serving.txt"
subprocess.check_call(["pip", "install", "-q", "-r", _req])
dbutils.library.restartPython()

In [0]:

import os
os.environ['ENABLE_MLFLOW_TRACING'] = "True"
model = "databricks-meta-llama-3-3-70b-instruct"
vector_search_endpoint = "pioneer_cdj"
catalog = "workspace"
db = "default"
vector_search_index = "pioneer_cdj_search_index"

chain_config = {
    "llm_model_serving_endpoint_name": model,  # the foundation model we want to use
    "vector_search_endpoint_name": vector_search_endpoint,  # the endoint we want to use for vector search
    "vector_search_index": f"{catalog}.{db}.{vector_search_index}",
    "llm_prompt_template": """You are a expert in pioneer cdj-3000X dj mixers meant to help the user learn the operation of this device. Some pieces of context may be irrelevant, in which case you should not use them to form the answer. If you do not know the answer to a question just say I do not know.  \n\nContext: {context}""",
}

In [0]:
from databricks.vector_search.client import VectorSearchClient
from databricks_langchain.vectorstores import DatabricksVectorSearch
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
import mlflow

## Enable MLflow Tracing
mlflow.langchain.autolog()

## Load the chain's configuration
model_config = mlflow.models.ModelConfig(development_config=chain_config)

## Turn the Vector Search index into a LangChain retriever
vector_search_as_retriever = DatabricksVectorSearch(
    endpoint=model_config.get("vector_search_endpoint_name"),
    index_name=model_config.get("vector_search_index"),
    columns=["id", "chunk_text"],
).as_retriever(search_kwargs={"k": 3})

# Method to format the docs returned by the retriever into the prompt (keep only the text from chunks)
def format_context(docs):
    chunk_contents = [f"Passage: {d.page_content}\n" for d in docs]
    return "".join(chunk_contents)

#Let's try our retriever chain:
relevant_docs = (vector_search_as_retriever | RunnableLambda(format_context)| StrOutputParser()).invoke('Is the pioneer cdj-3000x compatible with rekordbox software?')

displayHTML(relevant_docs)
     

In [0]:
from langchain_core.prompts import ChatPromptTemplate
from databricks_langchain.chat_models import ChatDatabricks
from operator import itemgetter
# Chat without vector retriever (pure llm case)
prompt = ChatPromptTemplate.from_messages(
    [  
        ("system", model_config.get("llm_prompt_template")), # Contains the instructions from the configuration
        ("user", "{question}") #user's questions
    ]
)

# Our foundation model answering the final prompt
model = ChatDatabricks(
    endpoint=model_config.get("llm_model_serving_endpoint_name"),
    extra_params={"temperature": 0.01, "max_tokens": 500}
)

#Let's try our prompt:
answer = (prompt | model | StrOutputParser()).invoke({'question':'How should you play a track on pioneer cdj-3000x?', 'context': ''})
displayHTML(answer)

In [0]:

from operator import itemgetter
# Chat with the vector retriever (rag case)

# Return the string contents of the most recent messages: [{...}] from the user to be used as input question
def extract_user_query_string(chat_messages_array):
    return chat_messages_array[-1]["content"]

# RAG Chain
chain = (
    {
        "question": itemgetter("messages") | RunnableLambda(extract_user_query_string),
        "context": itemgetter("messages")
        | RunnableLambda(extract_user_query_string)
        | vector_search_as_retriever
        | RunnableLambda(format_context),
    }
    | prompt
    | model
    | StrOutputParser()
    #| RunnableLambda(chat_completion_response_parser)
)

In [0]:

# Let's give it a try:
input_example = {"messages": [ {"role": "user", "content": 'How should you play a track on pioneer cdj-3000x?'}]}
answer = chain.invoke(input_example)
print(answer)

In [0]:
import yaml

rag_chain_config = {
    "databricks_resources": {
        "llm_endpoint_name": model_config.get("llm_model_serving_endpoint_name"),  
        "vector_search_endpoint_name": model_config.get("vector_search_endpoint_name"),
    },
    "input_example": {
       "messages": [ {"role": "user", "content": "How should you play a track on pioneer cdj-3000x?"}]
    },
    "llm_config": {
        "llm_parameters": {"max_tokens": 1500, "temperature": 0.01},
        "llm_prompt_template": model_config.get("llm_prompt_template"),
        "llm_prompt_template_variables": ["context"],
    },
    "retriever_config": {
        "chunk_template": "Passage: {chunk_text}\n",
        "parameters": {"k": 3},
        "schema": {"chunk_text": "chunk_text", "primary_key": "id"},
        "vector_search_index": model_config.get("vector_search_index"),
    }
}
try:
    with open('rag_chain_config.yaml', 'w') as f:
        yaml.dump(rag_chain_config, f)
except:
    print('pass to work on build job')
model_config = mlflow.models.ModelConfig(development_config='rag_chain_config.yaml')
     

In [0]:
%%writefile chain.py
from databricks.vector_search.client import VectorSearchClient
from databricks_langchain.vectorstores import DatabricksVectorSearch
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
import mlflow
# We need the py file saved in disk in order to serialize the model in Mlflow
## Enable MLflow Tracing
mlflow.langchain.autolog()

model_config = mlflow.models.ModelConfig(development_config="rag_chain_config.yaml")

# Turn the Vector Search index into a LangChain retriever
vector_search_as_retriever = DatabricksVectorSearch(
    endpoint=model_config.get("vector_search_endpoint_name"),
    index_name=model_config.get("vector_search_index"),
    columns=["id", "chunk_text"],
).as_retriever(search_kwargs={"k": 3}) # Number of search results that the retriever returns
# Enable the RAG Studio Review App and MLFlow to properly display track and display retrieved chunks for evaluation
mlflow.models.set_retriever_schema(primary_key="id", text_column="chunk_text")

# Method to format the docs returned by the retriever into the prompt (keep only the text from chunks)
def format_context(docs):
    chunk_contents = [f"Passage: {d.page_content}\n" for d in docs]
    return "".join(chunk_contents)

from langchain_core.prompts import ChatPromptTemplate
from databricks_langchain.chat_models import ChatDatabricks
from operator import itemgetter

prompt = ChatPromptTemplate.from_messages(
    [
        (  # System prompt contains the instructions
            "system",
            """You are a expert in pioneer cdj-3000X dj mixers meant to help the user learn the operation of this device. Some pieces of context may be irrelevant, in which case you should not use them to form the answer. If you do not know the answer to a question just say I do not know.  \n\nContext: {context}""",
        ),
        # User's question
        ("user", "{question}"),
    ]
)

# Our foundation model answering the final prompt
model = ChatDatabricks(
    endpoint=model_config.get("llm_model_serving_endpoint_name"),
    extra_params={"temperature": 0.01, "max_tokens": 500}
)

# Return the string contents of the most recent messages: [{...}] from the user to be used as input question
def extract_user_query_string(chat_messages_array):
    return chat_messages_array[-1]["content"]

# RAG Chain
chain = (
    {
        "question": itemgetter("messages") | RunnableLambda(extract_user_query_string),
        "context": itemgetter("messages")
        | RunnableLambda(extract_user_query_string)
        | vector_search_as_retriever
        | RunnableLambda(format_context),
    }
    | prompt
    | model
)


# Tell MLflow logging where to find your chain.
mlflow.models.set_model(model=chain)

In [0]:
from mlflow.models.resources import DatabricksVectorSearchIndex, DatabricksServingEndpoint
from mlflow.models.signature import infer_signature
import mlflow
import os
from mlflow.types.llm import ChatCompletionRequest

input_example = {"messages":[ {"role": "user", "content": "How should you play a track on pioneer cdj-3000x?"}]}


# Log the model to MLflow
with mlflow.start_run(run_name="rag_cdj_3000x"):
  logged_chain_info = mlflow.langchain.log_model(
          lc_model=os.path.join(os.getcwd(), 'chain.py'),
          name="chain",  
          model_config=chain_config,
          input_example=input_example,
          resources=[
            DatabricksVectorSearchIndex(index_name=model_config.get("retriever_config")["vector_search_index"]),
            DatabricksServingEndpoint(endpoint_name=model_config.get("databricks_resources")["llm_endpoint_name"])
          ]
      )



MODEL_NAME = "rag_cdj_3000x_chatbot"
MODEL_NAME_FQN = f"{catalog}.{db}.{MODEL_NAME}"
# Register to UC
uc_registered_model_info = mlflow.register_model(model_uri=logged_chain_info.model_uri, name=MODEL_NAME_FQN)


In [0]:
# This creates a functional endpoint but without the chat interface

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput

w = WorkspaceClient()

w.serving_endpoints.create(
    name="rag_cdj_endpoint_basic",
    config=EndpointCoreConfigInput(
        name="rag_cdj_endpoint_basic",
        served_entities=[
            ServedEntityInput(
                entity_name=MODEL_NAME_FQN,
                entity_version=uc_registered_model_info.version,
                workload_size="Small",
                scale_to_zero_enabled=True
            )
        ]
    )
)

In [0]:
from databricks.sdk import WorkspaceClient
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

# Initialize Databricks client
w = WorkspaceClient()

# Endpoint name
ENDPOINT_NAME = "rag_cdj_endpoint_basic"

# Chat history
chat_history = []

def query_endpoint(message):
    """Query the serving endpoint with a message"""
    try:
        input_data = {
            "messages": [
                {"role": "user", "content": message}
            ]
        }
        
        response = w.serving_endpoints.query(
            name=ENDPOINT_NAME,
            inputs=input_data
        )
        
        # Extract the content from predictions
        if hasattr(response, 'predictions') and len(response.predictions) > 0:
            # predictions[0] is a dict with 'content' key
            return response.predictions[0]['content']
        elif hasattr(response, 'choices') and len(response.choices) > 0:
            return response.choices[0].message.content
        else:
            return str(response)
    except Exception as e:
        return f"❌ Error: {str(e)}"

def display_chat():
    """Display the chat history"""
    html_content = '<div style="font-family: Arial; max-width: 800px;">'
    html_content += '<h2 style="color: #1976D2;">🎧 Pioneer CDJ-3000X Expert Assistant</h2>'
    
    for msg in chat_history:
        if msg['role'] == 'user':
            html_content += f'<div style="background-color: #E3F2FD; padding: 10px; margin: 10px 0; border-radius: 10px;"><strong>You:</strong> {msg["content"]}</div>'
        else:
            html_content += f'<div style="background-color: #F5F5F5; padding: 10px; margin: 10px 0; border-radius: 10px;"><strong>Assistant:</strong> {msg["content"]}</div>'
    
    html_content += '</div>'
    return HTML(html_content)

def on_send_clicked(b):
    """Handle send button click"""
    message = text_input.value.strip()
    if not message:
        return
    
    # Add user message to history
    chat_history.append({"role": "user", "content": message})
    
    # Clear input
    text_input.value = ""
    
    # Show loading
    with output:
        clear_output(wait=True)
        display(display_chat())
        display(HTML('<div style="color: #666; font-style: italic;">🤔 Thinking...</div>'))
    
    # Get response
    response = query_endpoint(message)
    
    # Add assistant response to history
    chat_history.append({"role": "assistant", "content": response})
    
    # Update display
    with output:
        clear_output(wait=True)
        display(display_chat())

def on_example_clicked(b):
    """Handle example button click"""
    text_input.value = b.description

# Create UI widgets
text_input = widgets.Textarea(
    placeholder='Ask me anything about the Pioneer CDJ-3000X...',
    layout=widgets.Layout(width='600px', height='60px')
)

send_button = widgets.Button(
    description='Send',
    button_style='primary',
    icon='paper-plane'
)
send_button.on_click(on_send_clicked)

clear_button = widgets.Button(
    description='Clear Chat',
    button_style='warning'
)

def on_clear_clicked(b):
    global chat_history
    chat_history = []
    with output:
        clear_output()
        display(HTML('<div style="font-family: Arial; max-width: 800px;"><h2 style="color: #1976D2;">🎧 Pioneer CDJ-3000X Expert Assistant</h2><p>Chat cleared. Ask me anything about the Pioneer CDJ-3000X!</p></div>'))

clear_button.on_click(on_clear_clicked)

# Example buttons
example1 = widgets.Button(description="How should you play a track on pioneer cdj-3000x?", layout=widgets.Layout(width='auto'))
example2 = widgets.Button(description="Is the pioneer cdj-3000x compatible with rekordbox software?", layout=widgets.Layout(width='auto'))
example3 = widgets.Button(description="What are the main features of the CDJ-3000X?", layout=widgets.Layout(width='auto'))
example4 = widgets.Button(description="How do I use the hot cues on the CDJ-3000X?", layout=widgets.Layout(width='auto'))

example1.on_click(on_example_clicked)
example2.on_click(on_example_clicked)
example3.on_click(on_example_clicked)
example4.on_click(on_example_clicked)

# Output area
output = widgets.Output()

# Layout
print("🚀 Chat interface loaded! Use the widgets below to chat.\n")

display(widgets.VBox([
    widgets.HTML('<h3>💬 Chat Input</h3>'),
    text_input,
    widgets.HBox([send_button, clear_button]),
    widgets.HTML('<h3>📝 Example Questions</h3>'),
    widgets.VBox([example1, example2, example3, example4]),
    output
]))

# Initial display
with output:
    display(HTML('<div style="font-family: Arial; max-width: 800px;"><h2 style="color: #1976D2;">🎧 Pioneer CDJ-3000X Expert Assistant</h2><p>Ask me anything about the Pioneer CDJ-3000X DJ mixer!</p></div>'))